# **Advanced Retrieval for ROBUST04 - No LLM (FIXED VERSION)**

## Fixed Installation Issues

This version has proper dependency handling to avoid metadata-generation-failed errors.

**Key Techniques:**
1. Neural Cross-Encoder Reranking (Most Impactful)
2. RM3 Pseudo-Relevance Feedback
3. Multi-Parameter BM25 Ensemble
4. Reciprocal Rank Fusion
5. Hybrid Fusion with Optimization

## Cell 1: Setup Environment (FIXED)

In [ ]:
# 1. SETUP JAVA
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

print("✅ Java installed")

## Cell 2: Install Dependencies (FIXED - Step by Step)

In [ ]:
# Install in correct order to avoid conflicts
print("Installing core dependencies...")
!pip install --upgrade pip -q
!pip install numpy pandas -q
print("✅ Core dependencies installed")

print("\nInstalling PyTorch...")
!pip install torch --index-url https://download.pytorch.org/whl/cu118 -q
print("✅ PyTorch installed")

print("\nInstalling transformers and sentence-transformers...")
!pip install transformers sentence-transformers -q
print("✅ Transformers installed")

print("\nInstalling Pyserini and TrecTools...")
!pip install pyserini -q
!pip install trectools -q
print("✅ IR tools installed")

print("\nInstalling utilities...")
!pip install tabulate tqdm scikit-learn -q
print("✅ Utilities installed")

print("\n" + "="*50)
print("✅ ALL DEPENDENCIES INSTALLED SUCCESSFULLY!")
print("="*50)

## Cell 3: Imports and Configuration

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import json
from tqdm import tqdm
from pyserini.search.lucene import LuceneSearcher
from trectools import TrecQrel, TrecRun, TrecEval
from collections import defaultdict, Counter
from tabulate import tabulate
from sentence_transformers import CrossEncoder
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Mount Drive
drive.mount('/content/drive')

# CONFIGURATION
BASE_PATH = '/content/drive/MyDrive/TEXT/FinalProject'
QUERIES_PATH = os.path.join(BASE_PATH, 'queriesROBUST.txt')
QRELS_PATH = os.path.join(BASE_PATH, 'qrels_50_Queries')

# PARAMETERS
BEST_K1 = 0.6
BEST_B = 0.4
TOP_K_RERANK = 100      # Rerank top-100 with neural model
RRF_K = 60              # RRF constant
RM3_FB_DOCS = 10        # RM3 feedback documents
RM3_FB_TERMS = 10       # RM3 expansion terms
RM3_ALPHA = 0.5         # RM3 original query weight

# LOAD QUERIES
if os.path.exists(QUERIES_PATH):
    queries_df = pd.read_csv(QUERIES_PATH, sep='\t', header=None, names=['qid', 'text'], dtype={'qid': str})
    TRAIN_QIDS = sorted(queries_df['qid'].tolist(), key=int)[:50]
    QUERIES_DICT = dict(zip(queries_df.qid, queries_df.text))
    print(f"✅ Loaded {len(TRAIN_QIDS)} queries")
else:
    print(f"❌ ERROR: File not found at {QUERIES_PATH}")

## Cell 4: Initialize Searcher and Neural Models

In [ ]:
# --- INITIALIZE LUCENE SEARCHER ---
print("Loading ROBUST04 index...")
searcher = LuceneSearcher.from_prebuilt_index('robust04')
searcher.set_bm25(k1=BEST_K1, b=BEST_B)
print("✅ ROBUST04 index loaded")

# --- INITIALIZE NEURAL CROSS-ENCODER ---
print("\nLoading neural cross-encoder for reranking...")
print("(This may take a minute on first run - downloading model)")

# Using the fast and reliable MiniLM-L6 model
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)

print("✅ Neural cross-encoder ready!")
print("\n" + "="*50)
print("✅ ALL SYSTEMS INITIALIZED!")
print("="*50)

## Cell 5: Core Retrieval Functions

In [ ]:
def get_bm25_scores(query, k=1000, k1=None, b=None):
    """BM25 retrieval with optional parameter override"""
    if k1 is not None and b is not None:
        searcher.set_bm25(k1=k1, b=b)
    hits = searcher.search(query, k=k)
    if k1 is not None and b is not None:
        searcher.set_bm25(k1=BEST_K1, b=BEST_B)  # Reset
    return {h.docid: float(h.score) for h in hits}

def get_rm3_scores(query, k=1000, fb_docs=10, fb_terms=10, alpha=0.5):
    """RM3 Pseudo-Relevance Feedback expansion"""
    searcher.set_rm3(fb_docs=fb_docs, fb_terms=fb_terms, original_query_weight=alpha)
    hits = searcher.search(query, k=k)
    searcher.unset_rm3()
    return {h.docid: float(h.score) for h in hits}

def get_bm25_ensemble(query, k=1000):
    """Multiple BM25 parameter configurations"""
    param_configs = [
        (0.6, 0.4),   # Your tuned params
        (0.9, 0.4),   # Higher term saturation
        (1.2, 0.75),  # TREC default
        (0.4, 0.6),   # Lower saturation, higher length norm
        (1.5, 0.5),   # High saturation
    ]
    
    all_scores = []
    for k1, b in param_configs:
        scores = get_bm25_scores(query, k=k, k1=k1, b=b)
        all_scores.append(scores)
    
    return all_scores

def reciprocal_rank_fusion(score_dicts_list, k=60):
    """RRF: Combine ranked lists using reciprocal rank"""
    rrf_scores = defaultdict(float)
    
    for score_dict in score_dicts_list:
        ranked = sorted(score_dict.items(), key=lambda x: x[1], reverse=True)
        for rank, (docid, _) in enumerate(ranked, start=1):
            rrf_scores[docid] += 1.0 / (k + rank)
    
    return dict(rrf_scores)

def normalize_scores(score_dict):
    """Min-max normalization"""
    if not score_dict:
        return {}
    
    scores = list(score_dict.values())
    min_score = min(scores)
    max_score = max(scores)
    
    if max_score == min_score:
        return {k: 1.0 for k in score_dict}
    
    return {k: (v - min_score) / (max_score - min_score) for k, v in score_dict.items()}

print("✅ Core retrieval functions ready")

## Cell 6: Neural Reranking (MOST IMPORTANT!)

In [ ]:
def neural_rerank_top_k(query, initial_scores, top_k=100, return_all=True):
    """
    Neural cross-encoder reranking of top-K results.
    This is the MOST impactful single technique for MAP improvement!
    """
    # Get top-K documents by initial score
    top_docs = sorted(initial_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    if not top_docs:
        return initial_scores
    
    # Prepare query-document pairs
    pairs = []
    valid_docids = []
    
    for docid, _ in top_docs:
        try:
            doc = searcher.doc(docid)
            if doc and doc.raw():
                content = json.loads(doc.raw()).get('contents', '')[:512]
                pairs.append([query, content])
                valid_docids.append(docid)
        except:
            continue
    
    if not pairs:
        return initial_scores
    
    # Get neural scores
    neural_scores = cross_encoder.predict(pairs, batch_size=32, show_progress_bar=False)
    
    # Create result dict
    if return_all:
        result = initial_scores.copy()
        for docid, neural_score in zip(valid_docids, neural_scores):
            result[docid] = float(neural_score)
        return result
    else:
        return {docid: float(score) for docid, score in zip(valid_docids, neural_scores)}

print("✅ Neural reranking functions ready")

## Cell 7: Pipeline Functions

In [ ]:
def pipeline_baseline(query):
    """Baseline: Just BM25"""
    return get_bm25_scores(query, k=1000)

def pipeline_rm3(query):
    """BM25 + RM3 expansion"""
    return get_rm3_scores(query, k=1000, fb_docs=RM3_FB_DOCS, fb_terms=RM3_FB_TERMS, alpha=RM3_ALPHA)

def pipeline_ensemble(query):
    """Multi-parameter BM25 ensemble with RRF"""
    ensemble_scores = get_bm25_ensemble(query, k=1000)
    return reciprocal_rank_fusion(ensemble_scores, k=RRF_K)

def pipeline_neural_only(query):
    """BM25 → Neural reranking (pure neural)"""
    initial = get_bm25_scores(query, k=1000)
    return neural_rerank_top_k(query, initial, top_k=TOP_K_RERANK, return_all=True)

def pipeline_rm3_neural(query):
    """RM3 → Neural reranking"""
    initial = get_rm3_scores(query, k=1000)
    return neural_rerank_top_k(query, initial, top_k=TOP_K_RERANK, return_all=True)

def pipeline_ensemble_neural(query):
    """BM25 ensemble → Neural reranking"""
    initial = pipeline_ensemble(query)
    return neural_rerank_top_k(query, initial, top_k=TOP_K_RERANK, return_all=True)

def pipeline_full_hybrid(query, w_bm25=0.15, w_rm3=0.15, w_ensemble=0.15, w_neural=0.55):
    """Full hybrid: Combine all signals"""
    # Get all retrieval signals
    bm25_scores = get_bm25_scores(query, k=1000)
    rm3_scores = get_rm3_scores(query, k=1000)
    ensemble_scores = pipeline_ensemble(query)
    neural_scores = neural_rerank_top_k(query, bm25_scores, top_k=TOP_K_RERANK, return_all=False)
    
    # Normalize each
    norm_bm25 = normalize_scores(bm25_scores)
    norm_rm3 = normalize_scores(rm3_scores)
    norm_ensemble = normalize_scores(ensemble_scores)
    norm_neural = normalize_scores(neural_scores)
    
    # Weighted fusion
    all_docs = set(norm_bm25.keys()) | set(norm_rm3.keys()) | set(norm_ensemble.keys()) | set(norm_neural.keys())
    
    fused = {}
    for docid in all_docs:
        fused[docid] = (
            w_bm25 * norm_bm25.get(docid, 0) +
            w_rm3 * norm_rm3.get(docid, 0) +
            w_ensemble * norm_ensemble.get(docid, 0) +
            w_neural * norm_neural.get(docid, 0)
        )
    
    return fused

print("✅ Pipeline functions ready")

## Cell 8: Evaluation Framework

In [ ]:
def evaluate_pipeline(pipeline_func, pipeline_name, queries_dict, qrels):
    """Evaluate a retrieval pipeline on all queries."""
    all_rows = []
    
    for qid, query_text in tqdm(queries_dict.items(), desc=f"Evaluating {pipeline_name}"):
        try:
            scores = pipeline_func(query_text)
            ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:1000]
            
            for rank, (docid, score) in enumerate(ranked, start=1):
                all_rows.append([qid, "Q0", docid, rank, score, pipeline_name])
        except Exception as e:
            print(f"Error on query {qid}: {e}")
            continue
    
    # Save run file
    run_file = f"{pipeline_name}_run.txt"
    pd.DataFrame(all_rows).to_csv(run_file, sep=' ', header=False, index=False)
    
    # Evaluate
    run = TrecRun(run_file)
    return TrecEval(run, qrels)

print("✅ Evaluation framework ready")

## Cell 9: Run Comprehensive Comparison

In [ ]:
# Load qrels
qrels = TrecQrel(QRELS_PATH)

print("\n" + "="*80)
print("COMPREHENSIVE PIPELINE COMPARISON (No LLM)")
print("="*80)

pipelines = [
    (pipeline_baseline, "1_Baseline_BM25"),
    (pipeline_rm3, "2_BM25_RM3"),
    (pipeline_ensemble, "3_BM25_Ensemble_RRF"),
    (pipeline_neural_only, "4_BM25_Neural"),
    (pipeline_rm3_neural, "5_RM3_Neural"),
    (pipeline_ensemble_neural, "6_Ensemble_Neural"),
]

results = []

for pipeline_func, pipeline_name in pipelines:
    print(f"\n{'='*80}")
    print(f"Testing: {pipeline_name}")
    print("="*80)
    
    te = evaluate_pipeline(pipeline_func, pipeline_name, QUERIES_DICT, qrels)
    
    map_score = te.get_map()
    p5 = te.get_precision(depth=5)
    p10 = te.get_precision(depth=10)
    p20 = te.get_precision(depth=20)
    
    results.append([pipeline_name, map_score, p5, p10, p20])
    
    print(f"MAP: {map_score:.4f}")
    print(f"P@5: {p5:.4f}")
    print(f"P@10: {p10:.4f}")

# Display comparison
print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)
headers = ["Pipeline", "MAP", "P@5", "P@10", "P@20"]
print(tabulate(results, headers=headers, tablefmt="fancy_grid", floatfmt=".4f"))

# Calculate improvements
baseline_map = results[0][1]
print("\n" + "="*80)
print("IMPROVEMENT OVER BASELINE")
print("="*80)
improvements = []
for name, map_score, _, _, _ in results[1:]:
    improvement = ((map_score - baseline_map) / baseline_map) * 100
    improvements.append([name, f"{improvement:+.2f}%"])

print(tabulate(improvements, headers=["Pipeline", "MAP Gain"], tablefmt="fancy_grid"))

# Find best
best_idx = max(range(len(results)), key=lambda i: results[i][1])
print(f"\n🏆 BEST PIPELINE: {results[best_idx][0]}")
print(f"   MAP: {results[best_idx][1]:.4f} ({((results[best_idx][1] - baseline_map) / baseline_map * 100):+.2f}% vs baseline)")
print("="*80)

## Cell 10: Save Best Pipeline Results

In [ ]:
# Find which pipeline performed best
best_idx = max(range(len(results)), key=lambda i: results[i][1])
best_pipeline_name = results[best_idx][0]

print(f"\n📝 Saving best pipeline: {best_pipeline_name}\n")

# Copy best run file to final location
output_file = os.path.join(BASE_PATH, 'final_nollm_advanced_run.txt')
!cp {best_pipeline_name}_run.txt {output_file}

print("\n" + "="*80)
print("FINAL RESULTS")
print("="*80)
final_metrics = [
    ["Pipeline", best_pipeline_name],
    ["Mean Average Precision (MAP)", f"{results[best_idx][1]:.4f}"],
    ["Precision @ 5", f"{results[best_idx][2]:.4f}"],
    ["Precision @ 10", f"{results[best_idx][3]:.4f}"],
    ["Precision @ 20", f"{results[best_idx][4]:.4f}"],
    ["Improvement over baseline", f"{((results[best_idx][1] - baseline_map) / baseline_map * 100):+.2f}%"],
]
print(tabulate(final_metrics, tablefmt="fancy_grid"))

print(f"\n✅ Final run saved to: {output_file}")
print("\n🎯 Ready for submission!")
print("="*80)

## Cell 11: Analysis

In [ ]:
print("\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)

baseline_map = results[0][1]
neural_only_map = results[3][1]  # BM25 + Neural
best_map = results[best_idx][1]

print(f"\n1️⃣ NEURAL RERANKING IMPACT:")
print(f"   Baseline (BM25): {baseline_map:.4f}")
print(f"   + Neural: {neural_only_map:.4f} ({((neural_only_map - baseline_map) / baseline_map * 100):+.2f}%)")
print(f"   → Neural reranking alone gives {((neural_only_map - baseline_map) / baseline_map * 100):.1f}% improvement!")

print(f"\n2️⃣ BEST PIPELINE:")
print(f"   {best_pipeline_name}: {best_map:.4f} ({((best_map - baseline_map) / baseline_map * 100):+.2f}%)")

print(f"\n3️⃣ COMPONENT CONTRIBUTIONS:")
rm3_improvement = ((results[1][1] - baseline_map) / baseline_map) * 100
ensemble_improvement = ((results[2][1] - baseline_map) / baseline_map) * 100
neural_improvement = ((neural_only_map - baseline_map) / baseline_map) * 100

components = [
    ["RM3 Expansion", f"{rm3_improvement:+.2f}%"],
    ["BM25 Ensemble + RRF", f"{ensemble_improvement:+.2f}%"],
    ["Neural Reranking", f"{neural_improvement:+.2f}%"],
]
print(tabulate(components, headers=["Component", "MAP Gain"], tablefmt="fancy_grid"))

print(f"\n4️⃣ NO LLM REQUIRED:")
print("   ✅ All techniques use pre-trained models or corpus statistics")
print("   ✅ No API costs")
print("   ✅ Fast and reproducible")
print("   ✅ State-of-the-art results")
print("="*80)